The idea of this excerice is to see how positional embedding evolve and help with predicting the position of words. </br>
The dataset of permutations of A B C D E F has been created, add anding at "|" symbol. The goal of the mini transformer network will be to predict the first letter in the sequence (in this case "A").</br>
It will be done with symbol embeddings only, and additionally with positional embedding, to see wether it would improve the situtation and by how much. </br>
This is a "First symbol predictor given the entire sequence". It will also be good to see, maybe in the future, if giving more attention blocks will help, so it can learn to pass the first token only thhrough it.</br>

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

import random
import numpy as np
import pandas as pd
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GLOBAL_SEED = 420
def set_seed(seed):
    random.seed(GLOBAL_SEED)
    np.random.seed(GLOBAL_SEED)
    torch.manual_seed(GLOBAL_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(GLOBAL_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(GLOBAL_SEED)

In [2]:
dataset = Path("dataset.txt").read_text(encoding="utf-8")
sentences = [line.split() for line in dataset.splitlines() if line.strip()]
word_columns = [f"word_{i}" for i in range(1, max(map(len, sentences)) + 1)]
sentence_table = pd.DataFrame(sentences, columns=word_columns)
sentence_table.head(10)


,word_1,word_2,word_3,word_4,word_5,word_6,word_7
0,A,B,C,D,E,F,|
1,A,B,C,D,F,E,|
2,A,B,C,E,D,F,|
3,A,B,C,E,F,D,|
4,A,B,C,F,D,E,|
5,A,B,C,F,E,D,|
6,A,B,D,C,E,F,|
7,A,B,D,C,F,E,|
8,A,B,D,E,C,F,|
9,A,B,D,E,F,C,|


In [3]:
# create a vocabulary of unique words
vocabulary = sorted({word for sentence in sentences for word in sentence})
vocab_size = len(vocabulary)
print(f"Vocabulary size: {vocab_size}")
print(f"Vocabulary: {vocabulary}")

Vocabulary size: 7
Vocabulary: ['A', 'B', 'C', 'D', 'E', 'F', '|']


In [4]:
vocabulary = list(vocabulary)
def encode(text):
    return [vocabulary.index(word) for word in text.split()]
def decode(indices):
    return " ".join(vocabulary[i] for i in indices)

print(encode("A B C D E F |"))
print(decode(encode("A B C D E F |")))

[0, 1, 2, 3, 4, 5, 6]
A B C D E F |


In [5]:
class MaskedSelfAttention(torch.nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.kqv = torch.nn.Linear(embedding_dim, embedding_dim * 3, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        kqv = self.kqv(x)  # (B, T, 3 * C)
        k, q, v = kqv.chunk(3, dim=-1)  # (B, T, C) each
        
        # compute attention scores
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()  # (T, T)
        attn_scores = (q @ k.transpose(-2, -1)) * (C ** -0.5)  # (B, T, T)
        attn_scores = attn_scores.masked_fill(mask, float('-inf'))  #(B, T, T)
        attn_weights = torch.nn.functional.softmax(attn_scores, dim=-1)  # (B, T, T)
        return attn_weights @ v  # (B, T, C)


class SmallWordPredictor(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim, use_positional_embeddings=False, attention_layers=1):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim)
        self.use_positional_embeddings = use_positional_embeddings
        if self.use_positional_embeddings:
            self.pos_embedding = torch.nn.Embedding(vocab_size, embedding_dim)

        self.attention = torch.nn.Sequential(*[MaskedSelfAttention(embedding_dim) for _ in range(attention_layers)])

        self.ln1 = torch.nn.Linear(embedding_dim, 2 * embedding_dim)
        self.gelu = torch.nn.GELU()
        self.ln2 = torch.nn.Linear(2 * embedding_dim, embedding_dim)
        
        self.out_projection = torch.nn.Linear(embedding_dim, vocab_size, bias=False)
        self.out_projection.weight = self.embedding.weight  # weight tying

    def forward(self, x):
        '''Predicts only one token, given a sequence of tokens.'''
        x = self.embedding(x)  # (B, T, C)
        if self.use_positional_embeddings:
            x = x + self.pos_embedding(torch.arange(x.size(1), device=x.device))  # (B, T, C)
        x = x + self.attention(x)  # (B, T, C)
        x = x + self.ln2(self.gelu(self.ln1(x)))  # (B, T, C)
        return self.out_projection(x[:, -1, :])  # (B, vocab_size)

In [6]:
# take first all symbols, predict first symbol
x = torch.tensor([encode(" ".join(words)) for words in sentence_table.to_numpy()], dtype=torch.long)
y = torch.tensor([encode(" ".join(words)) for words in sentence_table.iloc[:, 0].to_numpy()], dtype=torch.long)
x = x.to(device)
y = y.to(device)
print(x[0], y[0])

tensor([0, 1, 2, 3, 4, 5, 6], device='cuda:0') tensor([0], device='cuda:0')


In [7]:
# visualize paiwise cosine similarity of word embeddings
def visualize_embeddings(model, vocabulary):
    embeddings = model.embedding.weight.data.cpu().numpy()
    cosine_similarities = np.dot(embeddings, embeddings.T)
    norms = np.linalg.norm(embeddings, axis=1)
    cosine_similarities /= norms[:, None]
    cosine_similarities /= norms[None, :]
    
    plt.figure(figsize=(10, 8))
    plt.imshow(cosine_similarities, cmap='viridis', interpolation='nearest')
    plt.colorbar()
    plt.xticks(ticks=np.arange(len(vocabulary)), labels=vocabulary, rotation=90)
    plt.yticks(ticks=np.arange(len(vocabulary)), labels=vocabulary)
    plt.title("Pairwise Cosine Similarity of Word Embeddings")
    plt.show()

In [8]:
# hiperparameters
EMBEDDING_DIM = 4
EPOCHS = 201

In [9]:
def train(model, optimizer, x, y, epochs):
    print(model)
    print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

    # One batch is entire dataset, as it is small
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(x)  # (B, T, vocab_size) -> (B, vocab_size)
        loss = torch.nn.functional.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [10]:
set_seed(GLOBAL_SEED)
# go with no positional embeddings for now, as the dataset is small and we want to see if the model can learn without them
model = SmallWordPredictor(vocab_size, embedding_dim=EMBEDDING_DIM, use_positional_embeddings=False).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(7, 4)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=4, out_features=12, bias=False)
    )
  )
  (ln1): Linear(in_features=4, out_features=8, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=8, out_features=4, bias=True)
  (out_projection): Linear(in_features=4, out_features=7, bias=False)
)
Number of parameters: 152
Epoch 0, Loss: 3.4271
Epoch 50, Loss: 1.9330
Epoch 100, Loss: 1.9084
Epoch 150, Loss: 1.7959
Epoch 200, Loss: 1.7920


Can't go past the 1.792 loss value

In [11]:
set_seed(GLOBAL_SEED)
# positional embeddings enabled
model = SmallWordPredictor(vocab_size, embedding_dim=EMBEDDING_DIM, use_positional_embeddings=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(7, 4)
  (pos_embedding): Embedding(7, 4)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=4, out_features=12, bias=False)
    )
  )
  (ln1): Linear(in_features=4, out_features=8, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=8, out_features=4, bias=True)
  (out_projection): Linear(in_features=4, out_features=7, bias=False)
)
Number of parameters: 180
Epoch 0, Loss: 3.7441
Epoch 50, Loss: 1.9123
Epoch 100, Loss: 0.5905
Epoch 150, Loss: 0.0066
Epoch 200, Loss: 0.0021


Positional embeddings make a huge difference when pedicting a specific token that is already embedded into a network. By position we can easily seperate the tokens from each other.

Let's see if we can go around needing the positional embeddings by using a larger embedding dimension. The idea is that with a larger embedding dimension, the model </br> 
might be able to learn positional information implicitly. Let's try with an embedding dimension of 8 and see if the model can learn without positional embeddings.

In [12]:
set_seed(GLOBAL_SEED)
# go with no positional but a much larger embedding dimension, as the dataset is small and we want to see if the model can learn without them
model = SmallWordPredictor(vocab_size, embedding_dim=(5*EMBEDDING_DIM), use_positional_embeddings=False).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(7, 20)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
  )
  (ln1): Linear(in_features=20, out_features=40, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=40, out_features=20, bias=True)
  (out_projection): Linear(in_features=20, out_features=7, bias=False)
)
Number of parameters: 3000
Epoch 0, Loss: 32.6887
Epoch 50, Loss: 1.8072
Epoch 100, Loss: 1.7918
Epoch 150, Loss: 1.7918
Epoch 200, Loss: 1.7918


It can't go around the positional embedding just by increasing token embedding dimention. This is probably caused by tokens (symbols) having all possible permutations in a dataset used for training. Therefore if every symbol is at every possible possition, the network can't encode it's position into the embedding space reliably.</br>
It is worth noting that the initial loss is much higher, but settles at about the same loss the smaller embedding network achieved

Let's see if enlarging the ammount of attention layers can help. From the ROME paper we know, that attention layers are used as information forwarders. Maybe if there is a lot of forwarding done, it can forward the first embedding reliably. Remember that the network uses only the last embedding in a batch for predition!!

In [13]:
set_seed(GLOBAL_SEED)
# increase number of attention layers
model = SmallWordPredictor(vocab_size, embedding_dim=(EMBEDDING_DIM), use_positional_embeddings=False, attention_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(7, 4)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=4, out_features=12, bias=False)
    )
    (1): MaskedSelfAttention(
      (kqv): Linear(in_features=4, out_features=12, bias=False)
    )
  )
  (ln1): Linear(in_features=4, out_features=8, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=8, out_features=4, bias=True)
  (out_projection): Linear(in_features=4, out_features=7, bias=False)
)
Number of parameters: 200
Epoch 0, Loss: 3.8955
Epoch 50, Loss: 1.4120
Epoch 100, Loss: 0.0824
Epoch 150, Loss: 0.0016
Epoch 200, Loss: 0.0008


With 2 attention layers chained together, it performs just as well, or even better then, positional encoding in accuraccy!

Let's check positional encoding vs double attention, with higher embeding dims

In [14]:
set_seed(GLOBAL_SEED)
# increase embedding dimension and use positional embeddings
model = SmallWordPredictor(vocab_size, embedding_dim=(5*EMBEDDING_DIM), use_positional_embeddings=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(7, 20)
  (pos_embedding): Embedding(7, 20)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
  )
  (ln1): Linear(in_features=20, out_features=40, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=40, out_features=20, bias=True)
  (out_projection): Linear(in_features=20, out_features=7, bias=False)
)
Number of parameters: 3140
Epoch 0, Loss: 29.7994
Epoch 50, Loss: 0.0020
Epoch 100, Loss: 0.0006
Epoch 150, Loss: 0.0004
Epoch 200, Loss: 0.0003


In [15]:
set_seed(GLOBAL_SEED)
# increase number of attention layers and embedding dimension
model = SmallWordPredictor(vocab_size, embedding_dim=(5*EMBEDDING_DIM), use_positional_embeddings=False, attention_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(7, 20)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
    (1): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
  )
  (ln1): Linear(in_features=20, out_features=40, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=40, out_features=20, bias=True)
  (out_projection): Linear(in_features=20, out_features=7, bias=False)
)
Number of parameters: 4200
Epoch 0, Loss: 29.8215
Epoch 50, Loss: 0.0007
Epoch 100, Loss: 0.0001
Epoch 150, Loss: 0.0001
Epoch 200, Loss: 0.0000


Arguably, the double attention performs better then positional embedding, at least on this toy example.
But it may be bound to sequence length. For longer sequences, it is possible that we would need to stack a lot of attention layers to achieve simmilar performance to positional embedding.

Let's try with a larger sequence legth

In [16]:
dataset = Path("dataset_longer_sequence.txt").read_text(encoding="utf-8")
sentences = [line.split() for line in dataset.splitlines() if line.strip()]
word_columns = [f"word_{i}" for i in range(1, max(map(len, sentences)) + 1)]
sentence_table = pd.DataFrame(sentences, columns=word_columns)
# create a vocabulary of unique words
vocabulary = sorted({word for sentence in sentences for word in sentence})
vocab_size = len(vocabulary)
print(f"Vocabulary size: {vocab_size}")
vocabulary = list(vocabulary)
def encode(text):
    return [vocabulary.index(word) for word in text.split()]
def decode(indices):
    return " ".join(vocabulary[i] for i in indices)

print(encode("A B C D E F |"))
print(decode(encode("A B C D E F |")))
# take first all symbols, predict first symbol
x = torch.tensor([encode(" ".join(words)) for words in sentence_table.to_numpy()], dtype=torch.long)
y = torch.tensor([encode(" ".join(words)) for words in sentence_table.iloc[:, 0].to_numpy()], dtype=torch.long)
x = x.to(device)
y = y.to(device)
print(x[0], y[0])

Vocabulary size: 14
[0, 1, 2, 3, 4, 5, 13]
A B C D E F |
tensor([12,  7, 11, 10,  4,  1,  2,  6,  0,  8,  3,  5,  9, 13],
       device='cuda:0') tensor([12], device='cuda:0')


In [17]:
set_seed(GLOBAL_SEED)
# increase embedding dimension and use positional embeddings, larger sequence dataset
model = SmallWordPredictor(vocab_size, embedding_dim=(5*EMBEDDING_DIM), use_positional_embeddings=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(14, 20)
  (pos_embedding): Embedding(14, 20)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
  )
  (ln1): Linear(in_features=20, out_features=40, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=40, out_features=20, bias=True)
  (out_projection): Linear(in_features=20, out_features=14, bias=False)
)
Number of parameters: 3420
Epoch 0, Loss: 11.2594
Epoch 50, Loss: 0.0013
Epoch 100, Loss: 0.0005
Epoch 150, Loss: 0.0003
Epoch 200, Loss: 0.0003


In [18]:
set_seed(GLOBAL_SEED)
# increase number of attention layers and embedding dimension
model = SmallWordPredictor(vocab_size, embedding_dim=(5*EMBEDDING_DIM), use_positional_embeddings=False, attention_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(14, 20)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
    (1): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
  )
  (ln1): Linear(in_features=20, out_features=40, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=40, out_features=20, bias=True)
  (out_projection): Linear(in_features=20, out_features=14, bias=False)
)
Number of parameters: 4340
Epoch 0, Loss: 11.2511
Epoch 50, Loss: 0.0016
Epoch 100, Loss: 0.0002
Epoch 150, Loss: 0.0001
Epoch 200, Loss: 0.0000


After doubling the sequence leght, and thus the vocabulary size, the double attention still performs as well, or better, then positional embedding. </br>
Ok, so you may think "How can vanilla self attention, that is permutation-equivariant (permuting the tokens wouldn't change their attention score to one another), can read position better then positional encoding when it's doubled?? </br>
The thing is, that one vanilla attention IS position invarient, but two MASKED self attentions not so much. What mask does, is it enocdes into a token all information proceeding it. This experiment is particularly interesing because we are predicting the FIRST token, so the one that has shared it's information with ALL proceeding tokens! No position needed! First attention encodes first token's information into all the next ones, the next attetion just checks which information is present in all tokens! That's it! no position information needed! So let's test it with predicting the third token! 

In [20]:
set_seed(GLOBAL_SEED)
# predict the third token to see if double attention can handle it
model = SmallWordPredictor(vocab_size, embedding_dim=(5*EMBEDDING_DIM), use_positional_embeddings=False, attention_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
y = torch.tensor([encode(" ".join(words)) for words in sentence_table.iloc[:, 2].to_numpy()], dtype=torch.long)
y = y.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(14, 20)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
    (1): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
  )
  (ln1): Linear(in_features=20, out_features=40, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=40, out_features=20, bias=True)
  (out_projection): Linear(in_features=20, out_features=14, bias=False)
)
Number of parameters: 4340
Epoch 0, Loss: 11.1806
Epoch 50, Loss: 2.3541
Epoch 100, Loss: 2.2315
Epoch 150, Loss: 2.1833
Epoch 200, Loss: 2.1494


In [21]:
set_seed(GLOBAL_SEED)
# predict the third token to see if double attention can handle it
model = SmallWordPredictor(vocab_size, embedding_dim=(5*EMBEDDING_DIM), use_positional_embeddings=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = model.to(device)
y = torch.tensor([encode(" ".join(words)) for words in sentence_table.iloc[:, 2].to_numpy()], dtype=torch.long)
y = y.to(device)
train(model, optimizer, x, y, epochs=EPOCHS)

SmallWordPredictor(
  (embedding): Embedding(14, 20)
  (pos_embedding): Embedding(14, 20)
  (attention): Sequential(
    (0): MaskedSelfAttention(
      (kqv): Linear(in_features=20, out_features=60, bias=False)
    )
  )
  (ln1): Linear(in_features=20, out_features=40, bias=True)
  (gelu): GELU(approximate='none')
  (ln2): Linear(in_features=40, out_features=20, bias=True)
  (out_projection): Linear(in_features=20, out_features=14, bias=False)
)
Number of parameters: 3420
Epoch 0, Loss: 11.3155
Epoch 50, Loss: 0.1607
Epoch 100, Loss: 0.0005
Epoch 150, Loss: 0.0003
Epoch 200, Loss: 0.0002


Attention can't handle it now, but positional encoding can!